In [ ]:
# run this notebook after ../pca/6_analyze_pca_R
# use this notebook to remove giab from trimmed and quality controlled 
# after this, run 23_generate_family_files_R

In [ ]:
%%bash 

tail -n +2 giab_difficult_merged_oct27.bed | cut -f1-3 > giab_remove.bed

In [ ]:
%%bash

A_BED="aou_ibd2_to_remove.bed"
B_BED="giab_remove.bed"
OUTPUT="aou_ibd2_giab_removed.tsv"

# check if bedtools is available
if ! command -v bedtools &> /dev/null; then
    echo "Error: bedtools is not installed or not in PATH"
    exit 1
fi

# check if input files exist
if [ ! -f "$A_BED" ]; then
    echo "Error: $A_BED not found"
    exit 1
fi

if [ ! -f "$B_BED" ]; then
    echo "Error: $B_BED not found"
    exit 1
fi

# create temporary directory
TMPDIR="tmp"
trap "rm -rf $TMPDIR" EXIT
mkdir -p $TMPDIR

echo "Processing pairs..."

# get unique pairs
cut -f4 "$A_BED" | sort -u > "$TMPDIR/pairs.txt"

# create output file with header
echo -e "pair\tlength_filter" > "$OUTPUT"

# process each pair
while read -r pair; do
    # extract regions for this pair
    grep -w "$pair" "$A_BED" | cut -f1-3 > "$TMPDIR/pair_regions.bed"
    
    # use bedtools subtract to remove regions from b.bed
    bedtools subtract -a "$TMPDIR/pair_regions.bed" -b "$B_BED" > "$TMPDIR/subtracted.bed"
    
    # calculate total length (sum of end - start for each region)
    total_length=$(awk '{sum += $3 - $2} END {print sum+0}' "$TMPDIR/subtracted.bed")
    
    # write result
    echo -e "${pair}\t${total_length}" >> "$OUTPUT"
    
done < "$TMPDIR/pairs.txt"

echo "Done! Results written to $OUTPUT"

In [ ]:
%%bash

A_BED="ukb_ibd2_to_remove.bed"
B_BED="giab_remove.bed"
OUTPUT="ukb_ibd2_giab_removed.tsv"

# check if bedtools is available
if ! command -v bedtools &> /dev/null; then
    echo "Error: bedtools is not installed or not in PATH"
    exit 1
fi

# check if input files exist
if [ ! -f "$A_BED" ]; then
    echo "Error: $A_BED not found"
    exit 1
fi

if [ ! -f "$B_BED" ]; then
    echo "Error: $B_BED not found"
    exit 1
fi

# create temporary directory
TMPDIR="tmp"
trap "rm -rf $TMPDIR" EXIT
mkdir -p $TMPDIR

echo "Processing pairs..."

# get unique pairs
cut -f4 "$A_BED" | sort -u > "$TMPDIR/pairs.txt"

# create output file with header
echo -e "pair\tlength_filter" > "$OUTPUT"

# process each pair
while read -r pair; do
    # extract regions for this pair
    grep -w "$pair" "$A_BED" | cut -f1-3 > "$TMPDIR/pair_regions.bed"
    
    # use bedtools subtract to remove regions from b.bed
    bedtools subtract -a "$TMPDIR/pair_regions.bed" -b "$B_BED" > "$TMPDIR/subtracted.bed"
    
    # calculate total length (sum of end - start for each region)
    total_length=$(awk '{sum += $3 - $2} END {print sum+0}' "$TMPDIR/subtracted.bed")
    
    # write result
    echo -e "${pair}\t${total_length}" >> "$OUTPUT"
    
done < "$TMPDIR/pairs.txt"

echo "Done! Results written to $OUTPUT"